# L83 · Transformer 从零复现 — 多头注意力（Multi-Head Attention，MHA） + 位置编码（Positional Encoding，PE） + Feed-Forward 完整实现

**目标**：从头实现 `scaled_dot_product_attention`，理解 Q/K/V 的几何含义，掌握缩放因子（Scaling Factor） `√d_k` 的必要性和 multi-head 的作用。

🔗 **Aurora 连接**：本节实现的注意力核心是 `aurora/whisper/` 编码器和 Month 5 本地 LLM 解码器的共同基础模块；后续 Whisper 编码器将直接调用这里导出的 `scaled_dot_product_attention`。

Transformer 的核心直觉：每个位置的输出是对所有位置 **值（V）** 的加权求和，权重由当前位置的**查询（Q）**与所有位置的**键（K）**的相似度决定。缩放点积让相似度在高维空间里不会饱和，softmax 把相似度归一化成概率分布。这一机制天然支持并行计算，是取代 RNN 的关键。

In [ ]:
import numpy as np

## 1. Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V

设序列长度为 `seq`，特征维度为 `d_k`（键/查询维）和 `d_v`（值维）。

- **Q**（Query）：当前位置想要「查什么」
- **K**（Key）：每个位置提供的「索引标签」
- **V**（Value）：每个位置实际携带的信息

计算步骤：
```
scores  = Q @ Kᵀ          # (seq_q, seq_k)  点积相似度
scores /= sqrt(d_k)        # 缩放
weights = softmax(scores)  # 每行归一化为概率
output  = weights @ V      # (seq_q, d_v)  加权聚合
```

几何上，`Q[i] · K[j]` 是向量夹角的余弦乘以模长——Q 和 K 方向越接近，该位置权重越高。

---

### 具体例子：一词多义的消解

处理句子 *"The bank by the river lowered rates"*，当计算 **bank** 这个词的输出时：

| 角色 | 内容 |
|---|---|
| **Q（bank 的查询）** | bank 在查询上下文：我是金融机构还是河岸？ |
| **K[river]（river 的键）** | river 标记自己为自然地理词 |
| **K[rates]（rates 的键）** | rates 标记自己为金融词 |
| **V[river]（river 的值）** | river 的语义嵌入（含水体、地理等信息）|
| **V[rates]（rates 的值）** | rates 的语义嵌入（含利率、金融等信息）|

注意力权重 `w[bank→river]` 和 `w[bank→rates]` 都偏高，两者的 V 按权重叠加后，**bank 的输出向量融合了河岸和利率两种上下文信息**，下游层据此消歧。

**Q/K/V 分离的意义**：Q 和 K 决定"关注哪里"，V 决定"提取什么"。两者解耦允许模型独立学习注意力规则（Q·K）和内容投影（V）——若令 K=V，则注意力规则和内容绑死，表达能力大幅下降。

In [ ]:
# 演示：2个查询，3个键/值，d_k=4，d_v=6
np.random.seed(0)
Q_demo = np.random.randn(2, 4)   # (seq_q=2, d_k=4)
K_demo = np.random.randn(3, 4)   # (seq_k=3, d_k=4)
V_demo = np.random.randn(3, 6)   # (seq_k=3, d_v=6)

scores_demo = Q_demo @ K_demo.T  # (2, 3)
print('scores shape:', scores_demo.shape)
print('scores (未缩放):\n', scores_demo.round(3))

## 2. 为什么除以 √d_k？

设 Q、K 的分量是均值0方差1的独立随机变量，则点积 `Q · K = Σ Q_i K_i` 的方差为 `d_k`，标准差为 `√d_k`。

当 `d_k` 很大（例如 512），点积的绝对值会很大，进入 softmax 的饱和区（某一项接近 1，其余接近 0），梯度几乎为 0，训练停滞。

除以 `√d_k` 后方差归一化为 1，softmax 输入数量级可控：
```
Var(Q · K / √d_k) = d_k / d_k = 1
```

In [ ]:
# 演示：d_k 对 softmax 输出分布的影响
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(42)
for d_k in [4, 64, 512]:
    q = rng.standard_normal(d_k)
    k = rng.standard_normal((8, d_k))
    raw   = softmax(q @ k.T)
    scaled = softmax(q @ k.T / np.sqrt(d_k))
    print(f'd_k={d_k:4d} | 未缩放最大权重={raw.max():.3f}  缩放后最大权重={scaled.max():.3f}')

## 3. Multi-Head Attention

单头注意力只能学到一种「相关性」。Multi-head 的做法：

```
将 d_model 拆成 h 份，每份维度 d_k = d_model / h
head_i = Attention(Q·Wq_i, K·Wk_i, V·Wv_i)   # 独立的线性投影
output = Concat(head_0, ..., head_{h-1}) · Wo  # concat 再投影
```

每个头在不同子空间里做注意力，可以同时关注「语法依存」「语义相似」「位置临近」等不同模式，最后线性变换融合。

本节只实现单头核心 `scaled_dot_product_attention`；multi-head 的投影矩阵 `Wq/Wk/Wv/Wo` 将在 L84 中封装进 `MultiHeadAttention` 类。

In [ ]:
# 演示：multi-head 概念——把 d_model=16 拆成 h=2 个头
d_model, h = 16, 2
d_k = d_model // h
rng = np.random.default_rng(7)
x = rng.standard_normal((1, 5, d_model))  # (B=1, seq=5, d_model=16)

# 简化：用切片模拟投影（真实实现用 Wq @ x）
for i in range(h):
    Q_h = x[:, :, i*d_k:(i+1)*d_k]
    K_h = x[:, :, i*d_k:(i+1)*d_k]
    V_h = x[:, :, i*d_k:(i+1)*d_k]
    print(f'head {i}: Q shape {Q_h.shape}, d_k={d_k}')

## 4. ✏️ 实现 `scaled_dot_product_attention(Q, K, V, mask=None)`

**签名**：`Q, K, V: np.ndarray` shape `(B, seq_q, d_k)` / `(B, seq_k, d_k)` / `(B, seq_k, d_v)` → `np.ndarray` shape `(B, seq_q, d_v)`

**推理路线**：
1. `scores = Q @ K.transpose(0,2,1) / np.sqrt(Q.shape[-1])`  →  shape `(B, seq_q, seq_k)`
2. 如果 `mask` 不为 None，将 `mask==True` 的位置置为 `-np.inf`（softmax 后变 0）
3. `weights = softmax(scores, axis=-1)`  →  每行归一化，和为 1
4. `return weights @ V`  →  shape `(B, seq_q, d_v)`

**参考输入输出**：
```
B=2, seq_q=4, seq_k=6, d_k=8, d_v=16
Q: (2,4,8)   K: (2,6,8)   V: (2,6,16)
scores: (2,4,6)   weights: (2,4,6) 每行和=1
output: (2,4,16)
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, seq_q, d_k)
    K: (B, seq_k, d_k)
    V: (B, seq_k, d_v)
    mask: bool array (B, seq_q, seq_k), True 表示屏蔽该位置
    返回: (B, seq_q, d_v)
    """
    d_k = Q.shape[-1]
    # ✏️ TODO: 计算缩放点积 scores，shape (B, seq_q, seq_k)
    scores = None

    # ✏️ TODO: 如果 mask 不为 None，屏蔽对应位置（置 -inf）

    # ✏️ TODO: softmax(scores, axis=-1) → weights
    weights = None

    # ✏️ TODO: 返回 weights @ V
    return None

In [ ]:
# 形状检查（当 scaled_dot_product_attention 未实现时静默跳过）
np.random.seed(1)
Q = np.random.randn(2, 4, 8)
K = np.random.randn(2, 6, 8)
V = np.random.randn(2, 6, 16)

try:
    out = scaled_dot_product_attention(Q, K, V)
    if out is None:
        print('⬜ 请先实现 scaled_dot_product_attention 的 4 个 TODO 步骤')
    else:
        assert out.shape == (2, 4, 16), f'形状错误: {out.shape}'
        print('✅ output shape:', out.shape)

        # 权重行和验证
        d_k = Q.shape[-1]
        scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
        scores_s = scores - scores.max(axis=-1, keepdims=True)
        w = np.exp(scores_s); w /= w.sum(axis=-1, keepdims=True)
        assert np.allclose(w.sum(axis=-1), 1.0, atol=1e-6)
        print('✅ softmax 权重每行和为 1')

        # Causal mask 检查
        B, seq = 1, 4
        Q2 = np.random.randn(B, seq, 8)
        K2 = np.random.randn(B, seq, 8)
        V2 = np.random.randn(B, seq, 8)
        mask = np.triu(np.ones((B, seq, seq), dtype=bool), k=1)
        out_masked = scaled_dot_product_attention(Q2, K2, V2, mask=mask)
        assert out_masked.shape == (B, seq, 8)
        print('✅ causal mask 输出形状正确:', out_masked.shape)

        # Causal mask 语义验证：学生输出需与参考实现一致
        scores_m = Q2 @ K2.transpose(0, 2, 1) / np.sqrt(d_k)
        scores_m[mask] = -1e9          # 屏蔽未来位置（大负数代替 -inf）
        e_m = np.exp(scores_m - scores_m.max(axis=-1, keepdims=True))
        w_m = e_m / e_m.sum(axis=-1, keepdims=True)
        ref_masked = w_m @ V2
        # pos=0 只能 attend pos=0，其余权重应 ≈ 0
        assert w_m[0, 0, 1:].max() < 1e-6, \
            f'causal mask 参考权重不符合预期: {w_m[0, 0, 1:]}'
        assert np.allclose(out_masked, ref_masked, atol=1e-4), \
            'causal mask 未正确应用：学生输出与参考实现不一致'
        print('✅ causal mask 语义验证通过（未来位置权重 ≈ 0，输出与参考一致）')
except TypeError:
    print('⬜ 函数返回了非数组类型，请完成所有 TODO')

## 5. 参数实验：Q 方向对注意力权重的影响

固定 `K`（3个键）和 `V`，改变 `Q` 的方向，观察 `weights` 如何随 `Q` 与各 `K` 夹角变化。

**预期现象**：
- 将 `Q` 设为与 `K[0]` 完全平行（`Q = K[0]`）→ `weights[0]` 接近 1
- 将 `Q` 设为均匀方向（`Q` 与所有 `K` 夹角相同）→ `weights` 趋于均匀分布 `[1/3, 1/3, 1/3]`
- `d_k` 越大，未缩放时权重越极端（更尖锐）

In [ ]:
np.random.seed(3)
d_k = 8
K_exp = np.random.randn(1, 3, d_k)  # 3个键
V_exp = np.random.randn(1, 3, d_k)

# 实验1：Q 与 K[0] 平行
Q_parallel = K_exp[:, [0], :]  # shape (1,1,d_k)
out1 = scaled_dot_product_attention(Q_parallel, K_exp, V_exp)
scores1 = Q_parallel @ K_exp.transpose(0,2,1) / np.sqrt(d_k)
s1 = scores1 - scores1.max(axis=-1, keepdims=True)
w1 = np.exp(s1); w1 /= w1.sum(axis=-1, keepdims=True)
print('Q ∥ K[0] → weights:', w1[0,0].round(3))

# 实验2：Q 为零向量（均匀注意力）
Q_zero = np.zeros((1, 1, d_k))
scores2 = Q_zero @ K_exp.transpose(0,2,1) / np.sqrt(d_k)
s2 = scores2 - scores2.max(axis=-1, keepdims=True)
w2 = np.exp(s2); w2 /= w2.sum(axis=-1, keepdims=True)
print('Q = 0 (均匀)   → weights:', w2[0,0].round(3))

# 实验3：d_k 对尖锐度的影响
print('\n--- d_k 对尖锐度的影响（未缩放 vs 缩放）---')
for dk in [4, 64, 512]:
    rng2 = np.random.default_rng(99)
    q_ = rng2.standard_normal((1,1,dk))
    k_ = rng2.standard_normal((1,3,dk))
    sc_raw = q_ @ k_.transpose(0,2,1)
    sc_scl = sc_raw / np.sqrt(dk)
    def sw(s):
        s = s - s.max(axis=-1, keepdims=True)
        e = np.exp(s); return e / e.sum(axis=-1, keepdims=True)
    print(f'  d_k={dk:4d}: 未缩放max={sw(sc_raw)[0,0].max():.3f}  缩放max={sw(sc_scl)[0,0].max():.3f}')

## 6. 位置编码（Positional Encoding，PE）

注意力机制对序列位置天然无感知（它只看内容相似度）。PE 把位置信息加到词嵌入（word embedding）里：

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

- 每个位置 `pos` 对应一个唯一的 d_model 维向量
- 相邻位置的 PE 点积随距离平滑衰减，帮助模型学到相对位置
- 正弦余弦交替：偶数维用 sin，奇数维用 cos

输入 x 加上 PE 后，注意力可以区分同一词出现在不同位置的情况。

In [ ]:
import numpy as np

def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]            # (seq_len, 1)
    i   = np.arange(0, d_model, 2)[None, :]     # (1, d_model//2)
    denom = 10000 ** (i / d_model)
    pe[:, 0::2] = np.sin(pos / denom)           # 偶数维 sin
    pe[:, 1::2] = np.cos(pos / denom)           # 奇数维 cos
    return pe

# 验证
seq_len, d_model = 8, 16
pe = positional_encoding(seq_len, d_model)
assert pe.shape == (seq_len, d_model)
# 偶数列应为 sin：|sin|<=1
assert np.all(np.abs(pe[:, 0::2]) <= 1.0 + 1e-9)
# 奇数列应为 cos：|cos|<=1
assert np.all(np.abs(pe[:, 1::2]) <= 1.0 + 1e-9)
print(f'positional_encoding({seq_len}, {d_model}) shape:', pe.shape)
print(f'第 0 行（pos=0）：{np.round(pe[0, :8], 3)}')
print(f'第 1 行（pos=1）：{np.round(pe[1, :8], 3)}')
print('✅ PE 形状与值域验证通过')


## 7. 前馈网络（Feed-Forward Network，FFN）

Transformer 的每个 Block 在注意力层后接一个两层 MLP：

```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

- 中间维度（`d_ff`）通常是 `d_model` 的 4 倍，扩展后再压缩
- 对序列的每个位置**独立**应用（position-wise）
- 与注意力层配合：注意力负责跨位置通信，FFN 负责每位置的特征变换

Whisper 的 d_model=512，d_ff=2048（4×）；GPT-2 的 d_model=768，d_ff=3072（4×）。

In [ ]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

class FFN:
    def __init__(self, d_model, d_ff, rng=None):
        if rng is None:
            rng = np.random.default_rng(0)
        scale = np.sqrt(2.0 / d_model)
        self.W1 = rng.normal(0, scale, (d_model, d_ff))
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.normal(0, scale, (d_ff, d_model))
        self.b2 = np.zeros(d_model)

    def __call__(self, x):
        # x: (seq_len, d_model)  or  (d_model,)
        h = relu(x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2

# 验证
d_model, d_ff, seq_len = 16, 64, 5
ffn = FFN(d_model, d_ff)
x_in = np.random.randn(seq_len, d_model)
x_out = ffn(x_in)
assert x_out.shape == (seq_len, d_model), f'输出形状错误: {x_out.shape}'
print(f'FFN({d_model}, {d_ff}): ({seq_len},{d_model}) -> {x_out.shape}')
print('✅ FFN 形状验证通过（position-wise 变换，输入输出维度相同）')


## 8. 组合：Attention + PE + FFN = Transformer Block 的核心

```
x_emb = embedding(tokens) + positional_encoding(seq_len, d_model)
x_att = scaled_dot_product_attention(Q=x_emb, K=x_emb, V=x_emb)
x_out = ffn(x_att + x_emb)  # 残差连接（Residual Connection）
```

真实 Transformer Block 还包括 LayerNorm，这里省略。
下一节（L84）在此基础上实现 LoRA 低秩适配层。

In [ ]:
import numpy as np

# ── 参考实现（仅供验证对比，不替代学生函数）────────────────────────
def softmax_2d(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _ref_sdpa(Q, K, V):
    """参考实现，用于与学生实现对比，不替代 scaled_dot_product_attention"""
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    return softmax_2d(scores) @ V

# Mini transformer block 参数
rng = np.random.default_rng(42)
d_model, d_ff, seq = 16, 64, 5

x = rng.standard_normal((seq, d_model))
pe = positional_encoding(seq, d_model)
x_pe = x + pe                          # 加位置编码

# ── 综合验证：调用学生实现 scaled_dot_product_attention ───────────
try:
    x_att = scaled_dot_product_attention(x_pe[None], x_pe[None], x_pe[None])[0]
    x_res = x_att + x_pe               # 残差连接
    ffn = FFN(d_model, d_ff, rng)
    x_out = ffn(x_res) + x_res        # FFN + 残差

    assert x_out.shape == (seq, d_model), f'形状错误: {x_out.shape}'

    # 与参考实现比较（使用相同随机种子）
    ref_rng = np.random.default_rng(42)
    x_ref = ref_rng.standard_normal((seq, d_model))
    x_pe_ref = x_ref + pe
    x_att_ref = _ref_sdpa(x_pe_ref[None], x_pe_ref[None], x_pe_ref[None])[0]
    x_res_ref = x_att_ref + x_pe_ref
    ffn_ref = FFN(d_model, d_ff, ref_rng)
    x_out_ref = ffn_ref(x_res_ref) + x_res_ref
    assert np.allclose(x_out, x_out_ref, atol=1e-6), \
        '学生实现与参考实现不一致，请检查 scaled_dot_product_attention'

    print(f'Transformer Block: ({seq},{d_model}) -> ({seq},{d_model})')
    print('✅ Attention + PE + FFN 完整前向传播通过（调用学生实现）')
except NotImplementedError:
    print('⬜ 请先完成 scaled_dot_product_attention 的 4 个 TODO 步骤再运行验证')
except (TypeError, AttributeError) as e:
    print(f'⬜ 函数返回类型错误，请完成所有 TODO: {e}')


## 本课收束

`scaled_dot_product_attention` 以 `(B, seq_q, d_k)` 的 Q/K 和 `(B, seq_k, d_v)` 的 V 为输入，输出 `(B, seq_q, d_v)` 的上下文向量——每行是对所有 V 的加权平均。
缩放因子 `1/√d_k` 保持 softmax 输入的方差恒为 1，避免梯度消失，是训练稳定的关键。
该函数将直接嵌入 `aurora/whisper/attention.py` 的 `MultiHeadAttention` 模块，作为 Whisper 编码器和 Month 5 本地 LLM 解码器的共同基础。
下节（**L84**）将在 `nn.Linear` 旁并联低秩矩阵 `lora_B @ lora_A`，实现 LoRA 微调：用 ~3-6% 的可训练参数替代全量微调。

## ✏️ 闭卷推导检查格 — 多头注意力维度追踪

**规则：关闭上方所有格，先在 Markdown 中填表，再运行 Code 格验证。若输出与预期不符，请修改此 Markdown 格中的答案后重新运行——不要直接看 Code 格的输出再填表。**

**题目**：多头注意力参数：batch=1，seq_len=T=20，d_model=512，h=8 头，d_k=d_v=64

逐步追踪张量形状（填写空格）：

| 步骤 | 操作 | 输出形状 |
|------|------|---------|
| 1 | 输入 X | (1, 20, 512) |
| 2 | Q = X @ W_Q | (1, 20, ___) |
| 3 | reshape → 多头 Q | (1, ___, 20, ___) |
| 4 | scores = Q @ K^T / √d_k | (1, ___, 20, ___) |
| 5 | weights = softmax(scores) | (1, ___, 20, ___) |
| 6 | attn = weights @ V | (1, ___, 20, ___) |
| 7 | concat → (1, 20, ___) |  |
| 8 | output = concat @ W_O | (1, 20, ___) |

写出步骤 4（scores = Q @ K^T）的矩阵乘法：K^T 的形状是什么？为什么要除以 √d_k？

（在此处填表并写说明...）

In [ ]:
# 验证：运行后检查你的答案是否正确
# 若某行与预期不符，回到 Markdown 格修改答案后重新运行（不要先看输出）
import numpy as np

B, T, D_MODEL, H = 1, 20, 512, 8
D_K = D_MODEL // H   # 64

np.random.seed(0)
X  = np.random.randn(B, T, D_MODEL)
Wq = np.random.randn(D_MODEL, D_MODEL) * 0.02
Wk = np.random.randn(D_MODEL, D_MODEL) * 0.02
Wv = np.random.randn(D_MODEL, D_MODEL) * 0.02
Wo = np.random.randn(D_MODEL, D_MODEL) * 0.02

Q = (X @ Wq).reshape(B, T, H, D_K).transpose(0,2,1,3)
K = (X @ Wk).reshape(B, T, H, D_K).transpose(0,2,1,3)
V = (X @ Wv).reshape(B, T, H, D_K).transpose(0,2,1,3)
scores  = Q @ K.transpose(0,1,3,2) / np.sqrt(D_K)
weights = np.exp(scores - scores.max(-1, keepdims=True))
weights /= weights.sum(-1, keepdims=True)
attn    = weights @ V
concat  = attn.transpose(0,2,1,3).reshape(B, T, D_MODEL)
out     = concat @ Wo

expected = {
    "Q (after W_Q)":      (B, T, D_MODEL),
    "Q (multi-head)":     (B, H, T, D_K),
    "scores (Q @ K^T)":   (B, H, T, T),
    "weights (softmax)":  (B, H, T, T),
    "attn (w @ V)":       (B, H, T, D_K),
    "concat":             (B, T, D_MODEL),
    "output":             (B, T, D_MODEL),
}
actual = {
    "Q (after W_Q)":      (X @ Wq).shape,
    "Q (multi-head)":     Q.shape,
    "scores (Q @ K^T)":   scores.shape,
    "weights (softmax)":  weights.shape,
    "attn (w @ V)":       attn.shape,
    "concat":             concat.shape,
    "output":             out.shape,
}

all_ok = True
for name, exp in expected.items():
    act = actual[name]
    ok  = act == exp
    all_ok = all_ok and ok
    print(f"  {'✅' if ok else '❌'} {name}: {act}  (期望 {exp})")

if all_ok:
    print("
✅ 所有维度正确！")
else:
    print("
❌ 有维度不符——请回到 Markdown 格修改填表答案，不要直接抄输出")